In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets", "openpyxl", "tqdm"])
import re
import os, random, time
import numpy as np
from collections import Counter
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda': print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
OUTPUT_DIR = "/kaggle/working/"; os.makedirs(OUTPUT_DIR, exist_ok=True)

D=256; VOCAB_SIZE=30522; MAX_SEQ=64; HEADS=4; DROP=0.15
CBAM_BLOCKS=3; TEXT_ENC_LAYERS=2; TEXT_REFINE_LAYERS=2; FUSE_LAYERS=4
EPOCHS=30; BS=32; LR=3e-4; WD=1e-4; ES_PAT=16; LR_PAT=4; LR_FAC=0.5; MIN_LR=1e-6; VAL_SPLIT=0.15
MAX_ANS_VOCAB=300; MIN_ANS_FREQ=3

# ─────────────────────────────────────────────────────────────────────
# 4. VizWiz  (e.g., Eldon/VizWiz or any HF mirror)
# ─────────────────────────────────────────────────────────────────────
# ~31,000 QA pairs from blind users · real-world photos
# Highly noisy: unanswerable questions, poor image quality,
# answers from 10 crowd annotators. Very diverse open-ended answers
# (objects, colors, text reading, brands, counts, etc.)

def normalize_answer_vizwiz(ans: str) -> str:
    """Normalize VizWiz answers."""
    ans = ans.strip().lower()
    ans = re.sub(r'[^\w\s\-/.,]', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Unanswerable / unsuitable canonicalization ──
    unanswerable_set = {
        'unanswerable', 'unsuitable', 'unsuitable image',
        'not answerable', 'cant answer', 'cannot answer',
        'i dont know', 'i don\'t know', 'i do not know',
        'unable to answer', 'not sure', 'unclear',
        'unreadable', 'cannot be determined', 'cant be determined',
        'can not be determined', 'not clear', 'blurry',
        'too blurry', 'too dark', 'no answer', 'na', 'n/a',
        'cannot tell', 'cant tell', 'hard to tell',
        'impossible to tell', 'nothing', 'not possible',
    }
    if ans in unanswerable_set:
        return 'unanswerable'

    # ── Yes / No ──
    yes_set = {'yes', 'yes.', 'yeah', 'yep', 'y', 'correct', 'true',
               'yes it is', 'yes, it is', 'yea', 'ya'}
    no_set  = {'no', 'no.', 'nope', 'n', 'false', 'incorrect', 'negative',
               'no it is not', 'nah'}
    if ans in yes_set:
        return 'yes'
    if ans in no_set:
        return 'no'

    # ── Numeric canonicalization ──
    word_to_num = {'zero': '0', 'one': '1', 'two': '2', 'three': '3',
                   'four': '4', 'five': '5', 'six': '6', 'seven': '7',
                   'eight': '8', 'nine': '9', 'ten': '10',
                   'eleven': '11', 'twelve': '12', 'thirteen': '13',
                   'fourteen': '14', 'fifteen': '15', 'twenty': '20',
                   'thirty': '30', 'forty': '40', 'fifty': '50',
                   'hundred': '100'}
    if ans in word_to_num:
        return word_to_num[ans]

    # ── Color normalization (very common in VizWiz) ──
    color_map = {
        'blue': 'blue', 'light blue': 'blue', 'dark blue': 'blue',
        'navy': 'blue', 'navy blue': 'blue', 'royal blue': 'blue',
        'red': 'red', 'dark red': 'red', 'light red': 'red',
        'maroon': 'red', 'crimson': 'red', 'burgundy': 'red',
        'green': 'green', 'light green': 'green', 'dark green': 'green',
        'lime': 'green', 'olive': 'green',
        'yellow': 'yellow', 'light yellow': 'yellow', 'gold': 'yellow',
        'golden': 'yellow',
        'orange': 'orange',
        'pink': 'pink', 'light pink': 'pink', 'hot pink': 'pink',
        'magenta': 'pink',
        'purple': 'purple', 'violet': 'purple', 'lavender': 'purple',
        'brown': 'brown', 'tan': 'brown', 'beige': 'brown',
        'khaki': 'brown',
        'white': 'white', 'off white': 'white', 'cream': 'white',
        'ivory': 'white',
        'black': 'black', 'dark': 'black',
        'gray': 'gray', 'grey': 'gray', 'silver': 'gray',
        'light gray': 'gray', 'light grey': 'gray',
        'dark gray': 'gray', 'dark grey': 'gray',
    }
    if ans in color_map:
        return color_map[ans]

    # ── Common object synonyms ──
    object_map = {
        'cellphone': 'phone', 'cell phone': 'phone', 'mobile': 'phone',
        'mobile phone': 'phone', 'smartphone': 'phone', 'iphone': 'phone',
        'tv': 'television', 'television': 'television',
        'laptop': 'laptop', 'computer': 'laptop', 'notebook': 'laptop',
        'can': 'can', 'cans': 'can', 'tin': 'can',
        'bottle': 'bottle', 'bottles': 'bottle',
        'box': 'box', 'boxes': 'box', 'package': 'box',
        'shirt': 'shirt', 'tshirt': 'shirt', 't-shirt': 'shirt',
        't shirt': 'shirt', 'tee shirt': 'shirt',
        'pants': 'pants', 'trousers': 'pants', 'jeans': 'pants',
        'shoe': 'shoe', 'shoes': 'shoe', 'sneaker': 'shoe',
        'sneakers': 'shoe',
        'remote': 'remote', 'remote control': 'remote',
        'glasses': 'glasses', 'eyeglasses': 'glasses',
        'sunglasses': 'sunglasses',
        'soda': 'soda', 'pop': 'soda', 'soft drink': 'soda',
        'coke': 'coca cola', 'coca-cola': 'coca cola',
        'pepsi': 'pepsi', 'dr pepper': 'dr pepper',
        'cat': 'cat', 'cats': 'cat', 'kitten': 'cat',
        'dog': 'dog', 'dogs': 'dog', 'puppy': 'dog',
        'dollar': 'dollar', 'dollars': 'dollar',
        'cent': 'cent', 'cents': 'cent',
    }
    if ans in object_map:
        return object_map[ans]

    # ── Remove articles and fillers ──
    ans = re.sub(r'^(the|a|an|its|it is|this is|that is|it\'s|i think)\s+', '', ans)
    ans = re.sub(r'\s+', ' ', ans).strip()

    # ── Remove trailing period ──
    ans = ans.rstrip('.')

    return ans


# ── LOAD VizWiz ──
print("\n" + "="*60 + "\nLOADING VizWiz INTO RAM\n" + "="*60)
from datasets import load_dataset
ds = load_dataset('lmms-lab/VizWiz-VQA')

def majority_answer(answers):
    if not answers: return ''
    clean = []
    for a in answers:
        if isinstance(a, str): clean.append(a.strip().lower())
        elif isinstance(a, dict): clean.append(str(a.get('answer','')).strip().lower())
        else: clean.append(str(a).strip().lower())
    clean = [a for a in clean if a and a not in ('unanswerable','unsuitable')]
    return Counter(clean).most_common(1)[0][0] if clean else ''

all_samples = []
for s in tqdm(ds['val'], desc="VizWiz val"):
    try:
        img = s.get('image'); q = str(s.get('question',''))
        if not img or not q: continue
        answer = s.get('answer')
        if answer is not None: answer = str(answer).strip().lower()
        else: answer = majority_answer(s.get('answers', []))
        answer = normalize_answer_vizwiz(answer)
        if not answer or answer in ('unanswerable','unsuitable',''): continue
        all_samples.append({'image': np.array(img.convert('RGB').resize((224,224)), dtype=np.float32)/255.0, 'question': q, 'answer': answer})
    except: continue
del ds; random.shuffle(all_samples)
sp = int(len(all_samples)*0.8)
train_samples, test_samples = all_samples[:sp], all_samples[sp:]
print(f"  Total usable: {len(all_samples)}, Train: {len(train_samples)}, Test: {len(test_samples)}")

all_ans = [s['answer'] for s in all_samples]; counts = Counter(all_ans)
filtered = [(a,c) for a,c in counts.most_common() if c >= MIN_ANS_FREQ][:MAX_ANS_VOCAB]
answer_vocab = {'<unk>': 0}
for i, (a,_) in enumerate(sorted(filtered, key=lambda x: x[0])): answer_vocab[a] = i+1
num_classes = len(answer_vocab)
cov = sum(counts[a] for a in answer_vocab if a in counts) / len(all_ans) * 100
print(f"  Vocab: {num_classes} classes, coverage: {cov:.1f}%")

indices = list(range(len(train_samples))); random.shuffle(indices)
n_val = int(len(indices)*VAL_SPLIT)
trn = [train_samples[i] for i in indices[n_val:]]; val = [train_samples[i] for i in indices[:n_val]]
print(f"  Train: {len(trn)}, Val: {len(val)}, Test: {len(test_samples)}")

def tokenize(questions):
    ids_l, mask_l = [], []
    for q in questions:
        w = q.lower().split()[:MAX_SEQ-2]
        ids = [1]+[hash(x)%(VOCAB_SIZE-2)+2 for x in w]+[2]; m = [1.0]*len(ids)
        while len(ids)<MAX_SEQ: ids.append(0); m.append(0.0)
        ids_l.append(ids[:MAX_SEQ]); mask_l.append(m[:MAX_SEQ])
    return ids_l, mask_l

class VQADataset(Dataset):
    def __init__(self, samples, vocab, augment=False):
        self.samples=samples; self.vocab=vocab; self.augment=augment
        self.ids, self.masks = tokenize([s['question'] for s in samples])
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s=self.samples[idx]; img=torch.tensor(s['image']).permute(2,0,1)
        if self.augment and random.random()>0.5: img=img.flip(-1)
        return img, torch.tensor(self.ids[idx],dtype=torch.long), torch.tensor(self.masks[idx],dtype=torch.float32), self.vocab.get(s['answer'],0)

def collate_fn(batch):
    imgs,ids,masks,lbls=zip(*batch)
    return torch.stack(imgs), torch.stack(ids), torch.stack(masks), torch.tensor(lbls,dtype=torch.long)

train_loader = DataLoader(VQADataset(trn,answer_vocab,True), batch_size=BS, shuffle=True, num_workers=2, pin_memory=True, collate_fn=collate_fn)
val_loader = DataLoader(VQADataset(val,answer_vocab), batch_size=BS, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)
test_loader = DataLoader(VQADataset(test_samples,answer_vocab), batch_size=BS, shuffle=False, num_workers=2, pin_memory=True, collate_fn=collate_fn)

# ── MODEL (same custom architecture) ──
class TransformerBlock(nn.Module):
    def __init__(s,dim,n_heads=4,ffn_ratio=4,dropout=0.1):
        super().__init__(); s.norm1=nn.LayerNorm(dim); s.norm2=nn.LayerNorm(dim)
        s.attn=nn.MultiheadAttention(dim,n_heads,dropout=dropout,batch_first=True)
        s.ffn=nn.Sequential(nn.Linear(dim,dim*ffn_ratio),nn.GELU(),nn.Dropout(dropout),nn.Linear(dim*ffn_ratio,dim),nn.Dropout(dropout))
    def forward(s,x,mask=None):
        h=s.norm1(x); kpm=(mask==0) if mask is not None else None; h,_=s.attn(h,h,h,key_padding_mask=kpm); x=x+h; return x+s.ffn(s.norm2(x))

class VisionEncoder(nn.Module):
    def __init__(s,dim=256):
        super().__init__()
        s.conv1=nn.Conv2d(3,32,7,stride=2,padding=3,bias=False); s.bn1=nn.BatchNorm2d(32); s.pool1=nn.MaxPool2d(3,stride=2,padding=1)
        s.conv2=nn.Conv2d(32,64,3,stride=2,padding=1,bias=False); s.bn2=nn.BatchNorm2d(64)
        s.conv3=nn.Conv2d(64,128,3,stride=2,padding=1,bias=False); s.bn3=nn.BatchNorm2d(128)
        s.conv4=nn.Conv2d(128,dim,3,stride=2,padding=1,bias=False); s.bn4=nn.BatchNorm2d(dim); s.norm=nn.LayerNorm(dim)
    def forward(s,x):
        h=s.pool1(F.silu(s.bn1(s.conv1(x)))); h=F.silu(s.bn2(s.conv2(h))); h=F.silu(s.bn3(s.conv3(h))); h=F.silu(s.bn4(s.conv4(h)))
        B,C,H,W=h.shape; return s.norm(h.permute(0,2,3,1).reshape(B,H*W,C))

class TextEncoder(nn.Module):
    def __init__(s,vocab_size=30522,dim=256,n_layers=2,n_heads=4,max_len=64,dropout=0.1):
        super().__init__(); s.tok_embed=nn.Embedding(vocab_size,dim); s.pos_embed=nn.Parameter(torch.randn(1,max_len,dim)*0.02)
        s.embed_norm=nn.LayerNorm(dim); s.embed_drop=nn.Dropout(dropout)
        s.blocks=nn.ModuleList([TransformerBlock(dim,n_heads,dropout=dropout) for _ in range(n_layers)]); s.final_norm=nn.LayerNorm(dim)
    def forward(s,input_ids,mask=None):
        L=input_ids.shape[1]; x=s.tok_embed(input_ids)+s.pos_embed[:,:L,:]; x=s.embed_drop(s.embed_norm(x))
        for blk in s.blocks: x=blk(x,mask=mask)
        return s.final_norm(x)

class ChannelAttention(nn.Module):
    def __init__(s,ch,r=8): super().__init__(); s.fc1=nn.Linear(ch,ch//r,bias=False); s.fc2=nn.Linear(ch//r,ch,bias=False)
    def forward(s,x): a=x.mean([1,2],keepdim=True); m=x.amax([1,2],keepdim=True); return x*torch.sigmoid(s.fc2(F.silu(s.fc1(a)))+s.fc2(F.silu(s.fc1(m))))

class SpatialAttention(nn.Module):
    def __init__(s): super().__init__(); s.c1=nn.Conv2d(2,8,3,padding=1,bias=False); s.c2=nn.Conv2d(2,8,3,padding=2,dilation=2,bias=False); s.fuse=nn.Conv2d(16,1,1,bias=False)
    def forward(s,x): xp=x.permute(0,3,1,2); a=xp.mean(1,keepdim=True); m=xp.amax(1,keepdim=True); c=torch.cat([a,m],1); return x*torch.sigmoid(s.fuse(torch.cat([s.c1(c),s.c2(c)],1))).permute(0,2,3,1)

class CBAMBlock(nn.Module):
    def __init__(s,ch): super().__init__(); s.ca=ChannelAttention(ch); s.sa=SpatialAttention(); s.ffn=nn.Sequential(nn.Linear(ch,ch*2),nn.GELU(),nn.Linear(ch*2,ch)); s.n1=nn.LayerNorm(ch); s.n2=nn.LayerNorm(ch)
    def forward(s,t): B,N,C=t.shape; sp=s.sa(s.ca(t.reshape(B,7,7,C))); t=s.n1(t+sp.reshape(B,N,C)); return s.n2(t+s.ffn(t))

class FusionLayer(nn.Module):
    def __init__(s,dim,n_heads,dropout):
        super().__init__()
        s.v2t=nn.MultiheadAttention(dim,n_heads,dropout=dropout,batch_first=True); s.v2t_n=nn.LayerNorm(dim); s.v2t_f=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(dropout),nn.Linear(dim*4,dim)); s.v2t_fn=nn.LayerNorm(dim)
        s.t2v=nn.MultiheadAttention(dim,n_heads,dropout=dropout,batch_first=True); s.t2v_n=nn.LayerNorm(dim); s.t2v_f=nn.Sequential(nn.Linear(dim,dim*4),nn.GELU(),nn.Dropout(dropout),nn.Linear(dim*4,dim)); s.t2v_fn=nn.LayerNorm(dim)
    def forward(s,v,t,kpm=None):
        o,_=s.v2t(v,t,t,key_padding_mask=kpm); v=s.v2t_n(v+o); v=s.v2t_fn(v+s.v2t_f(v))
        o,_=s.t2v(t,v,v); t=s.t2v_n(t+o); t=s.t2v_fn(t+s.t2v_f(t)); return v,t

class MedicalVQAModel(nn.Module):
    def __init__(s,nc):
        super().__init__()
        s.vision_enc=VisionEncoder(D); s.text_enc=TextEncoder(VOCAB_SIZE,D,TEXT_ENC_LAYERS,HEADS,MAX_SEQ,DROP)
        s.vis_refine=nn.ModuleList([CBAMBlock(D) for _ in range(CBAM_BLOCKS)])
        s.text_refine=nn.ModuleList([TransformerBlock(D,HEADS,dropout=DROP) for _ in range(TEXT_REFINE_LAYERS)]); s.text_refine_norm=nn.LayerNorm(D)
        s.q_attn=nn.MultiheadAttention(D,HEADS,dropout=DROP,batch_first=True); s.q_gate=nn.Linear(D,D); s.q_norm=nn.LayerNorm(D)
        s.fusion_layers=nn.ModuleList([FusionLayer(D,HEADS,DROP) for _ in range(FUSE_LAYERS)])
        s.pool_query=nn.Parameter(torch.randn(1,1,D)*0.02); s.pool_attn=nn.MultiheadAttention(D,HEADS,dropout=DROP,batch_first=True); s.pool_norm=nn.LayerNorm(D)
        s.hfc1=nn.Linear(D,D); s.hd1=nn.Dropout(DROP); s.hfc2=nn.Linear(D,D//2); s.hd2=nn.Dropout(DROP); s.hout=nn.Linear(D//2,nc); s.hres=nn.Linear(D,D//2); s.hn=nn.LayerNorm(D//2)
    def forward(s,images,input_ids,text_mask):
        v=s.vision_enc(images); t=s.text_enc(input_ids,mask=text_mask)
        for b in s.vis_refine: v=b(v)
        for b in s.text_refine: t=b(t,mask=text_mask)
        t=s.text_refine_norm(t)
        qc=t[:,0:1,:].expand(-1,v.shape[1],-1); ao,_=s.q_attn(qc,v,v); g=torch.sigmoid(s.q_gate(ao)); v=s.q_norm(v+v*g+ao*(1-g))
        kpm=(text_mask==0)
        for fl in s.fusion_layers: v,t=fl(v,t,kpm=kpm)
        comb=torch.cat([v,t],1); B=comb.shape[0]; pq=s.pool_query.expand(B,-1,-1); p,_=s.pool_attn(pq,comb,comb); f=s.pool_norm(pq+p).squeeze(1)
        h=s.hd1(F.gelu(s.hfc1(f))); h=s.hd2(F.gelu(s.hfc2(h))); return s.hout(s.hn(h+s.hres(f)))

model = MedicalVQAModel(num_classes).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"\n  Model: {n_params:,} ({n_params/1e6:.1f}M)")

# ── TRAINING ──
print("\n" + "="*60 + "\nTRAINING (Centralized)\n" + "="*60)
criterion = nn.CrossEntropyLoss(); optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
history = {'epoch':[],'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[],'test_loss':[],'test_acc':[],'lr':[],'epoch_time':[]}
best_val, best_state, pat, lr_pat = 0.0, None, 0, 0

@torch.no_grad()
def ev(loader):
    model.eval(); ls,c,t=0.0,0,0
    for i,d,m,l in loader: i,d,m,l=i.to(device),d.to(device),m.to(device),l.to(device); lo=model(i,d,m); loss=criterion(lo,l); ls+=loss.item()*l.size(0); c+=(lo.argmax(-1)==l).sum().item(); t+=l.size(0)
    return ls/t, 100*c/t

for epoch in range(1,EPOCHS+1):
    t0=time.time(); model.train(); tl,tc,tt=0.0,0,0
    pb=tqdm(train_loader,desc=f"E{epoch:02d}/{EPOCHS}",leave=False)
    for i,d,m,l in pb:
        i,d,m,l=i.to(device),d.to(device),m.to(device),l.to(device); optimizer.zero_grad(); lo=model(i,d,m); loss=criterion(lo,l)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        tl+=loss.item()*l.size(0); tc+=(lo.argmax(-1)==l).sum().item(); tt+=l.size(0); pb.set_postfix(l=f"{loss.item():.3f}",a=f"{100*tc/tt:.1f}%")
    tl/=tt; ta=100*tc/tt; vl,va=ev(val_loader); tel,tea=ev(test_loader); lr=optimizer.param_groups[0]['lr']; et=time.time()-t0
    history['epoch'].append(epoch); history['train_loss'].append(tl); history['train_acc'].append(ta); history['val_loss'].append(vl); history['val_acc'].append(va)
    history['test_loss'].append(tel); history['test_acc'].append(tea); history['lr'].append(lr); history['epoch_time'].append(et)
    mk=""
    if va>best_val: best_val=va; best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}; pat=lr_pat=0; mk=" ★"
    else: pat+=1; lr_pat+=1
    print(f"E{epoch:02d} [{et:.1f}s]  Train: {tl:.4f}/{ta:.1f}%  Val: {vl:.4f}/{va:.1f}%  Test: {tea:.1f}%  LR={lr:.1e}{mk}")
    if lr_pat>=LR_PAT:
        for pg in optimizer.param_groups: pg['lr']=max(pg['lr']*LR_FAC,MIN_LR); lr_pat=0
    if pat>=ES_PAT: print(f"  Early stop"); break

if best_state: model.load_state_dict(best_state)
tel,tea=ev(test_loader); print(f"\n{'='*60}\nFINAL TEST: {tea:.2f}%\n{'='*60}")

# ── SAVE EXCEL ──
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
wb=openpyxl.Workbook(); ws=wb.active; ws.title="Training"
hf=Font(name='Arial',bold=True,size=11,color='FFFFFF'); hfi=PatternFill(start_color='1A5276',end_color='1A5276',fill_type='solid')
headers=['Epoch','Train Loss','Train Acc (%)','Val Loss','Val Acc (%)','Test Loss','Test Acc (%)','LR','Time (s)']
for c,h in enumerate(headers,1): cl=ws.cell(row=1,column=c,value=h); cl.font=hf; cl.fill=hfi; cl.alignment=Alignment(horizontal='center')
for i,ep in enumerate(history['epoch']):
    r=i+2
    for c,k in enumerate(['epoch','train_loss','train_acc','val_loss','val_acc','test_loss','test_acc','lr','epoch_time'],1):
        v=history[k][i]; ws.cell(row=r,column=c,value=round(v,4) if isinstance(v,float) else v)
ws2=wb.create_sheet("Summary")
for i,(k,v) in enumerate([("Method","Centralized"),("Dataset","VizWiz"),("Architecture","Custom CNN+CBAM+CrossAttn"),("Params",f"{n_params:,}"),("Classes",num_classes),("Best Val",round(best_val,2)),("Final Test",round(tea,2))],1):
    ws2.cell(row=i,column=1,value=k).font=Font(bold=True,name='Arial'); ws2.cell(row=i,column=2,value=v)
for s in [ws,ws2]:
    for col in s.columns: s.column_dimensions[col[0].column_letter].width=max(len(str(c.value or '')) for c in col)+2
p=f"{OUTPUT_DIR}/vizwiz_centralized_custom_results.xlsx"; wb.save(p); print(f"\nSaved → {p}\nDONE!")

Device: cuda
GPU: Tesla T4, VRAM: 15.6 GB

LOADING VizWiz INTO RAM


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00008-8bb04d0d8c47d4a(…):   0%|          | 0.00/494M [00:00<?, ?B/s]

data/test-00001-of-00008-0deaf01822797f0(…):   0%|          | 0.00/530M [00:00<?, ?B/s]

data/test-00002-of-00008-a4468dc23224a35(…):   0%|          | 0.00/482M [00:00<?, ?B/s]

...

E24 [10.3s]  Train: 1.7956/64.2%  Val: 2.3491/57.2%  Test: 62.6%  LR=3.7e-05


E25/30:   0%|          | 0/86 [00:00<?, ?it/s]

E25 [39.6s]  Train: 1.7833/64.4%  Val: 2.3609/55.2%  Test: 61.4%  LR=3.7e-05


E26/30:   0%|          | 0/86 [00:00<?, ?it/s]

E26 [9.7s]  Train: 1.7743/64.6%  Val: 2.3601/55.8%  Test: 62.3%  LR=3.7e-05
  Early stop

FINAL TEST: 63.12%

Saved → /kaggle/working//vizwiz_centralized_custom_results.xlsx
DONE!
